# 🌳 Árbol de expresión: construcción, recorridos y graficación

**Ejemplo base:** `sqrt(2*(x^2)+5*x-3)`, es decir, $\sqrt{2x^2 + 5x - 3}$

Este cuaderno recibe **cualquier expresión matemática** y:

1. **Valida** la expresión (caracteres, paréntesis, operadores, funciones, números).
2. La **tokeniza** (la separa en números, variables, operadores, funciones y paréntesis).
3. La convierte a **notación postfija** con el algoritmo *Shunting-Yard* (Dijkstra), mostrando la tabla paso a paso.
4. **Construye el árbol de expresión** a partir de la postfija usando una pila.
5. Obtiene los recorridos **preorden**, **inorden** y **postorden**.
6. **Dibuja el árbol** y, si hay una sola variable, **grafica la función**.
7. **Evalúa** la expresión para valores dados, validando errores de dominio.

**Cómo usarlo:** `Entorno de ejecución → Ejecutar todas` (o `Ctrl + F9`).
Para probar otra expresión, cambia la variable `EXPRESION` de la celda 9 o usa el modo interactivo (celda 11).

## Celda 1 · Librerías y configuración

Se definen las funciones, constantes y operadores reconocidos.
Cada operador tiene **precedencia**, **asociatividad** y **aridad** (número de operandos):

| Operador | Significado | Precedencia | Asociatividad | Aridad |
|---|---|---|---|---|
| `+` `-` | suma, resta | 1 | izquierda | 2 |
| `*` `/` | multiplicación, división | 2 | izquierda | 2 |
| `neg` | negativo unario (`-x`) | 3 | derecha | 1 |
| `^` | potencia | 4 | derecha | 2 |

> `^` es asociativo por la derecha: `2^3^2 = 2^(3^2) = 512`.
> El negativo unario tiene menor precedencia que `^`: `-x^2 = -(x^2)`.

In [ ]:
import re
import math
from dataclasses import dataclass

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

# Funciones de un argumento reconocidas
FUNCIONES = {
    'sqrt': math.sqrt,   # raíz cuadrada
    'sin':  math.sin,    # seno (radianes)
    'cos':  math.cos,    # coseno (radianes)
    'tan':  math.tan,    # tangente (radianes)
    'log':  math.log10,  # logaritmo base 10
    'ln':   math.log,    # logaritmo natural
    'exp':  math.exp,    # e^x
    'abs':  abs,         # valor absoluto
}

# Constantes reconocidas
CONSTANTES = {'pi': math.pi, 'e': math.e}

# operador: (precedencia, asociatividad, aridad)
OPERADORES = {
    '+':   (1, 'izq', 2),
    '-':   (1, 'izq', 2),
    '*':   (2, 'izq', 2),
    '/':   (2, 'izq', 2),
    'neg': (3, 'der', 1),
    '^':   (4, 'der', 2),
}

TIPOS_OPERANDO = ('NUM', 'VAR', 'CONST')


class ErrorExpresion(Exception):
    """Error de sintaxis o de evaluación en una expresión."""
    pass


print("✅ Configuración cargada.")
print("   Funciones:", ', '.join(FUNCIONES))
print("   Constantes:", ', '.join(CONSTANTES))
print("   Operadores: + - * / ^  (y - unario)")

## Celda 2 · Normalización y tokenización (análisis léxico)

**Normalizar** unifica la escritura: `**` → `^`, `√` → `sqrt`, `×` → `*`, `[ ]` y `{ }` → `( )`, coma decimal → punto, mayúsculas → minúsculas.

**Tokenizar** recorre la cadena carácter por carácter y produce una lista de *tokens*.
Aquí ocurren las primeras validaciones: caracteres no permitidos, números mal formados e identificadores desconocidos.

In [ ]:
@dataclass
class Token:
    tipo: str    # NUM, VAR, CONST, FUNC, OP, PAR_IZQ, PAR_DER
    valor: str
    pos: int     # posición en la cadena, para señalar errores


def _senalar(expr, pos, mensaje):
    """Arma un mensaje de error con una flecha bajo la posición del problema."""
    return f"{mensaje}\n      {expr}\n      {' ' * pos}^"


def normalizar(expr):
    if not isinstance(expr, str):
        raise ErrorExpresion("La expresión debe ser un texto (str).")
    expr = expr.strip()
    if not expr:
        raise ErrorExpresion("La expresión está vacía.")
    if expr.count('=') > 0:
        raise ErrorExpresion("Escribe solo una expresión, sin el signo '=' "
                             "(por ejemplo: 2*x^2+5*x-3 en lugar de y=2*x^2+5*x-3).")
    reemplazos = [('**', '^'), ('×', '*'), ('·', '*'), ('÷', '/'), ('−', '-'),
                  ('√', ' sqrt '), ('[', '('), (']', ')'), ('{', '('), ('}', ')'),
                  (',', '.')]
    for viejo, nuevo in reemplazos:
        expr = expr.replace(viejo, nuevo)
    expr = re.sub(r'\s+', ' ', expr).strip().lower()
    return expr


def tokenizar(expr):
    tokens = []
    i, n = 0, len(expr)
    while i < n:
        c = expr[i]

        if c.isspace():
            i += 1
            continue

        # Números: 12, 3.5, .5, 2.
        if c.isdigit() or c == '.':
            texto = re.match(r'\d*\.?\d*', expr[i:]).group()
            fin = i + len(texto)
            if texto == '.':
                raise ErrorExpresion(_senalar(expr, i, "Punto decimal sin dígitos."))
            if fin < n and expr[fin] == '.':
                raise ErrorExpresion(_senalar(expr, fin, f"Número mal formado (más de un punto decimal)."))
            tokens.append(Token('NUM', texto, i))
            i = fin
            continue

        # Identificadores: funciones, constantes o variables
        if c.isascii() and c.isalpha():
            palabra = re.match(r'[a-z]+', expr[i:]).group()
            if palabra in FUNCIONES:
                tokens.append(Token('FUNC', palabra, i))
            elif palabra in CONSTANTES:
                tokens.append(Token('CONST', palabra, i))
            elif len(palabra) == 1:
                tokens.append(Token('VAR', palabra, i))
            else:
                raise ErrorExpresion(_senalar(
                    expr, i,
                    f"Identificador desconocido '{palabra}'.\n"
                    f"      Funciones válidas: {', '.join(FUNCIONES)}. Constantes: {', '.join(CONSTANTES)}.\n"
                    f"      Las variables son de una letra; para multiplicar variables escribe x*y."))
            i += len(palabra)
            continue

        if c in '+-*/^':
            tokens.append(Token('OP', c, i))
        elif c == '(':
            tokens.append(Token('PAR_IZQ', c, i))
        elif c == ')':
            tokens.append(Token('PAR_DER', c, i))
        else:
            raise ErrorExpresion(_senalar(expr, i, f"Carácter no válido '{c}'."))
        i += 1
    return tokens


print("✅ Tokenizador listo.")

## Celda 3 · Preprocesamiento y validación sintáctica

**Preprocesar:**
- Detecta el **signo menos unario** (`-x`, `2*-3`, `(-5)`) y lo convierte en el operador `neg`. El `+` unario se descarta porque no cambia el valor.
- Inserta **multiplicaciones implícitas**: `2x` → `2*x`, `3(x+1)` → `3*(x+1)`, `(x+1)(x-1)` → `(x+1)*(x-1)`, `2sqrt(x)` → `2*sqrt(x)`.

**Validar** con una pequeña máquina de estados que alterna entre *"espero un operando"* y *"espero un operador"*. Detecta:
operadores seguidos, expresión que empieza o termina con operador, paréntesis vacíos o desbalanceados,
funciones sin paréntesis y números pegados sin operador.

In [ ]:
def _simbolo(valor):
    return '-' if valor == 'neg' else valor


def preprocesar(tokens, expr):
    """Detecta signos unarios e inserta multiplicaciones implícitas."""
    resultado, avisos = [], []
    for tok in tokens:
        prev = resultado[-1] if resultado else None

        # 1) Multiplicación implícita
        if prev is not None:
            termina_operando = prev.tipo in TIPOS_OPERANDO + ('PAR_DER',)
            inicia_operando = tok.tipo in TIPOS_OPERANDO + ('FUNC', 'PAR_IZQ')
            if termina_operando and inicia_operando:
                if prev.tipo == 'NUM' and tok.tipo == 'NUM':
                    raise ErrorExpresion(_senalar(
                        expr, tok.pos, f"Falta un operador entre {prev.valor} y {tok.valor}."))
                resultado.append(Token('OP', '*', tok.pos))
                avisos.append(f"Multiplicación implícita insertada: {prev.valor} * {tok.valor}")
                prev = resultado[-1]

        # 2) Signo unario: al inicio, después de un operador o después de '('
        if tok.tipo == 'OP' and tok.valor in ('+', '-'):
            if prev is None or prev.tipo in ('OP', 'PAR_IZQ'):
                if tok.valor == '+':
                    continue
                tok = Token('OP', 'neg', tok.pos)

        resultado.append(tok)
    return resultado, avisos


def validar_sintaxis(tokens, expr):
    if not tokens:
        raise ErrorExpresion("La expresión no contiene elementos válidos.")

    espera_operando = True
    parentesis_abiertos = []          # posiciones de '(' sin cerrar

    for i, tok in enumerate(tokens):
        anterior = tokens[i - 1] if i > 0 else None
        siguiente = tokens[i + 1] if i + 1 < len(tokens) else None

        if espera_operando:
            if tok.tipo in TIPOS_OPERANDO:
                espera_operando = False
            elif tok.tipo == 'PAR_IZQ':
                parentesis_abiertos.append(tok.pos)
            elif tok.tipo == 'FUNC':
                if siguiente is None or siguiente.tipo != 'PAR_IZQ':
                    raise ErrorExpresion(_senalar(
                        expr, tok.pos,
                        f"La función '{tok.valor}' debe ir seguida de '('. Ejemplo: {tok.valor}(x)"))
            elif tok.tipo == 'OP' and tok.valor == 'neg':
                pass                      # el negativo unario sigue esperando operando
            elif tok.tipo == 'PAR_DER':
                if anterior is not None and anterior.tipo == 'PAR_IZQ':
                    msg = "Paréntesis vacíos '()'."
                else:
                    msg = "Falta un operando antes de ')'."
                raise ErrorExpresion(_senalar(expr, tok.pos, msg))
            else:                         # operador binario donde se esperaba operando
                if anterior is None:
                    msg = f"La expresión no puede iniciar con el operador '{tok.valor}'."
                elif anterior.tipo == 'OP':
                    msg = f"Dos operadores seguidos: '{_simbolo(anterior.valor)}' y '{tok.valor}'."
                else:
                    msg = f"Falta un operando antes de '{tok.valor}'."
                raise ErrorExpresion(_senalar(expr, tok.pos, msg))
        else:
            if tok.tipo == 'OP':
                espera_operando = True
            elif tok.tipo == 'PAR_DER':
                if not parentesis_abiertos:
                    raise ErrorExpresion(_senalar(
                        expr, tok.pos, "Paréntesis ')' sin su '(' correspondiente."))
                parentesis_abiertos.pop()
            else:
                raise ErrorExpresion(_senalar(
                    expr, tok.pos, f"Falta un operador antes de '{tok.valor}'."))

    if espera_operando:
        ultimo = tokens[-1]
        raise ErrorExpresion(_senalar(
            expr, ultimo.pos,
            f"La expresión está incompleta: termina en '{_simbolo(ultimo.valor)}'."))
    if parentesis_abiertos:
        faltan = len(parentesis_abiertos)
        raise ErrorExpresion(_senalar(
            expr, parentesis_abiertos[-1],
            f"Falta{'n' if faltan > 1 else ''} {faltan} paréntesis de cierre ')'."))


print("✅ Validador listo.")

## Celda 4 · Conversión infija → postfija (algoritmo Shunting-Yard)

Reglas:
- **Operando** → pasa directo a la salida.
- **Función** o **`(`** → se apila.
- **Operador binario** → antes de apilarlo, se sacan a la salida los operadores de la pila con **mayor** precedencia
  (o **igual** precedencia si el operador es asociativo por la izquierda).
- **Operador unario** (`neg`) → se apila directamente (es prefijo, no tiene operando izquierdo).
- **`)`** → se desapila hacia la salida hasta encontrar `(`; si debajo hay una función, también sale.
- **Fin** → se vacía la pila.

In [ ]:
def _texto_tokens(lista):
    return ' '.join(t.valor for t in lista) if lista else '∅'


def a_postfija(tokens):
    salida, pila, pasos = [], [], []

    def anotar(tok, accion):
        pasos.append((tok, accion, _texto_tokens(pila), _texto_tokens(salida)))

    for tok in tokens:
        if tok.tipo in TIPOS_OPERANDO:
            salida.append(tok)
            anotar(tok.valor, "Operando → salida")

        elif tok.tipo in ('FUNC', 'PAR_IZQ'):
            pila.append(tok)
            anotar(tok.valor, "Función → pila" if tok.tipo == 'FUNC' else "'(' → pila")

        elif tok.tipo == 'PAR_DER':
            while pila and pila[-1].tipo != 'PAR_IZQ':
                salida.append(pila.pop())
            if not pila:
                raise ErrorExpresion("Paréntesis desbalanceados.")
            pila.pop()                                   # descarta '('
            accion = "Desapilar hasta '('"
            if pila and pila[-1].tipo == 'FUNC':
                salida.append(pila.pop())
                accion += " y sacar la función"
            anotar(')', accion)

        elif tok.tipo == 'OP':
            prec, asoc, aridad = OPERADORES[tok.valor]
            sacados = 0
            if aridad == 2:
                while pila and pila[-1].tipo == 'OP':
                    prec_tope = OPERADORES[pila[-1].valor][0]
                    if prec_tope > prec or (prec_tope == prec and asoc == 'izq'):
                        salida.append(pila.pop())
                        sacados += 1
                    else:
                        break
            pila.append(tok)
            accion = f"Operador → pila" + (f" (antes salen {sacados})" if sacados else "")
            anotar(tok.valor, accion)

    while pila:
        tope = pila.pop()
        if tope.tipo == 'PAR_IZQ':
            raise ErrorExpresion("Paréntesis desbalanceados.")
        salida.append(tope)
    anotar('fin', "Vaciar la pila → salida")
    return salida, pasos


def imprimir_pasos(pasos):
    ancho_pila = max(len("Pila"), *(len(p[2]) for p in pasos))
    encabezado = f"{'#':>3} │ {'Token':<6} │ {'Acción':<40} │ {'Pila':<{ancho_pila}} │ Salida"
    print(encabezado)
    print('─' * len(encabezado) + '─' * 20)
    for k, (tok, accion, pila, salida) in enumerate(pasos, 1):
        print(f"{k:>3} │ {tok:<6} │ {accion:<40} │ {pila:<{ancho_pila}} │ {salida}")


print("✅ Shunting-Yard listo.")

## Celda 5 · Nodo y construcción del árbol

Se recorre la expresión **postfija** con una pila:
- **Operando** → se crea una hoja y se apila.
- **Operador binario** → se desapilan dos nodos: el primero es el hijo **derecho** y el segundo el **izquierdo**.
- **Función** o **`neg`** (unarios) → se desapila un nodo, que queda como hijo **derecho**
  (así el inorden se lee naturalmente: `sqrt` y luego su argumento).

Al final la pila debe contener **exactamente un nodo**: la raíz.

In [ ]:
class Nodo:
    def __init__(self, valor, tipo, izq=None, der=None):
        self.valor = valor   # texto que se muestra en el nodo
        self.tipo = tipo     # NUM, VAR, CONST, OP (binario), OP_UN (neg), FUNC
        self.izq = izq
        self.der = der

    def es_hoja(self):
        return self.izq is None and self.der is None

    def __repr__(self):
        return f"Nodo({self.valor!r})"


def construir_arbol(postfija):
    pila = []
    for tok in postfija:
        if tok.tipo in TIPOS_OPERANDO:
            pila.append(Nodo(tok.valor, tok.tipo))
        elif tok.tipo == 'FUNC' or tok.valor == 'neg':
            if not pila:
                raise ErrorExpresion(f"Falta el operando de '{tok.valor}'.")
            hijo = pila.pop()
            tipo = 'FUNC' if tok.tipo == 'FUNC' else 'OP_UN'
            pila.append(Nodo(tok.valor, tipo, der=hijo))
        else:
            if len(pila) < 2:
                raise ErrorExpresion(f"Faltan operandos para '{tok.valor}'.")
            der = pila.pop()
            izq = pila.pop()
            pila.append(Nodo(tok.valor, 'OP', izq, der))
    if len(pila) != 1:
        raise ErrorExpresion("Expresión mal formada: sobran o faltan operandos.")
    return pila[0]


print("✅ Constructor del árbol listo.")

## Celda 6 · Recorridos, estadísticas y evaluación

| Recorrido | Orden de visita | Notación que produce |
|---|---|---|
| **Preorden** | raíz → izquierdo → derecho | prefija (polaca) |
| **Inorden** | izquierdo → raíz → derecho | infija (la usual, sin paréntesis) |
| **Postorden** | izquierdo → derecho → raíz | postfija (polaca inversa) |

El inorden **pierde los paréntesis**, por eso también se reconstruye la expresión **infija con paréntesis**,
que sí conserva el significado.

La **evaluación** recorre el árbol en postorden (primero los hijos, luego el operador) y valida
errores de dominio: raíz de negativo, logaritmo de cero o negativo, división entre cero, potencias complejas y desbordamiento.

In [ ]:
def preorden(n):
    if n is None:
        return []
    return [n.valor] + preorden(n.izq) + preorden(n.der)


def inorden(n):
    if n is None:
        return []
    return inorden(n.izq) + [n.valor] + inorden(n.der)


def postorden(n):
    if n is None:
        return []
    return postorden(n.izq) + postorden(n.der) + [n.valor]


def infija_con_parentesis(n):
    if n.tipo in TIPOS_OPERANDO:
        return n.valor
    if n.tipo == 'FUNC':
        return f"{n.valor}({infija_con_parentesis(n.der)})"
    if n.tipo == 'OP_UN':
        return f"(-{infija_con_parentesis(n.der)})"
    return f"({infija_con_parentesis(n.izq)} {n.valor} {infija_con_parentesis(n.der)})"


def altura(n):
    return 0 if n is None else 1 + max(altura(n.izq), altura(n.der))


def contar_nodos(n):
    return 0 if n is None else 1 + contar_nodos(n.izq) + contar_nodos(n.der)


def contar_hojas(n):
    if n is None:
        return 0
    return 1 if n.es_hoja() else contar_hojas(n.izq) + contar_hojas(n.der)


def variables(n):
    if n is None:
        return set()
    propia = {n.valor} if n.tipo == 'VAR' else set()
    return propia | variables(n.izq) | variables(n.der)


def evaluar(n, valores=None):
    valores = valores or {}
    if n.tipo == 'NUM':
        return float(n.valor)
    if n.tipo == 'CONST':
        return CONSTANTES[n.valor]
    if n.tipo == 'VAR':
        if n.valor not in valores:
            raise ErrorExpresion(f"Falta el valor de la variable '{n.valor}'.")
        return float(valores[n.valor])

    if n.tipo == 'OP_UN':
        return -evaluar(n.der, valores)

    if n.tipo == 'FUNC':
        a = evaluar(n.der, valores)
        if n.valor == 'sqrt' and a < 0:
            raise ErrorExpresion(f"Error de dominio: sqrt({a:g}) — raíz cuadrada de un número negativo.")
        if n.valor in ('log', 'ln') and a <= 0:
            raise ErrorExpresion(f"Error de dominio: {n.valor}({a:g}) — el logaritmo requiere un valor mayor que 0.")
        if n.valor == 'tan' and abs(math.cos(a)) < 1e-12:
            raise ErrorExpresion(f"Error de dominio: tan({a:g}) no está definida.")
        try:
            r = FUNCIONES[n.valor](a)
        except OverflowError:
            raise ErrorExpresion(f"Desbordamiento al calcular {n.valor}({a:g}).")
        return r

    a = evaluar(n.izq, valores)
    b = evaluar(n.der, valores)
    try:
        if n.valor == '+':
            r = a + b
        elif n.valor == '-':
            r = a - b
        elif n.valor == '*':
            r = a * b
        elif n.valor == '/':
            if b == 0:
                raise ErrorExpresion(f"División entre cero: {a:g} / 0.")
            r = a / b
        elif n.valor == '^':
            if a == 0 and b < 0:
                raise ErrorExpresion(f"División entre cero: 0^{b:g}.")
            if a < 0 and not float(b).is_integer():
                raise ErrorExpresion(f"Error de dominio: ({a:g})^{b:g} da un número complejo.")
            r = a ** b
        else:
            raise ErrorExpresion(f"Operador desconocido '{n.valor}'.")
    except OverflowError:
        raise ErrorExpresion(f"Desbordamiento al calcular {a:g} {n.valor} {b:g}.")
    if not math.isfinite(r):
        raise ErrorExpresion(f"El resultado de {a:g} {n.valor} {b:g} no es un número finito.")
    return r


print("✅ Recorridos y evaluación listos.")

## Celda 7 · Graficación del árbol y de la función

- El árbol se dibuja con `matplotlib`: la **coordenada x** de cada nodo es su posición en el recorrido **inorden**
  y la **coordenada y** es su **profundidad**, de modo que ningún nodo se encima.
- Los colores distinguen el tipo de nodo, y cada arista indica si es hijo izquierdo (**I**) o derecho (**D**).
- Si la expresión tiene **una sola variable**, también se grafica `f(variable)`; los puntos fuera del dominio se omiten.

In [ ]:
COLORES = {
    'OP':    ('#4C72B0', 'Operador binario'),
    'OP_UN': ('#8172B3', 'Operador unario (neg)'),
    'FUNC':  ('#DD8452', 'Función'),
    'NUM':   ('#55A868', 'Número'),
    'VAR':   ('#C44E52', 'Variable'),
    'CONST': ('#2A9D8F', 'Constante'),
}


def dibujar_arbol(raiz, titulo="Árbol de expresión", guardar_como=None):
    posiciones, nodos = {}, []
    contador = [0]

    def asignar(n, prof):
        if n is None:
            return
        asignar(n.izq, prof + 1)
        posiciones[id(n)] = (contador[0], -prof)
        nodos.append(n)
        contador[0] += 1
        asignar(n.der, prof + 1)

    asignar(raiz, 0)

    # Los nodos unarios (funciones y neg) se centran justo encima de su único hijo
    def centrar_unarios(n):
        if n is None:
            return
        centrar_unarios(n.izq)
        centrar_unarios(n.der)
        if n.tipo in ('FUNC', 'OP_UN'):
            posiciones[id(n)] = (posiciones[id(n.der)][0], posiciones[id(n)][1])

    centrar_unarios(raiz)
    h = altura(raiz)
    fig, ax = plt.subplots(figsize=(max(7, 1.0 * len(nodos)), max(4, 1.3 * h + 1)))

    # Aristas
    for n in nodos:
        x0, y0 = posiciones[id(n)]
        for hijo, lado in ((n.izq, 'I'), (n.der, 'D')):
            if hijo is not None:
                x1, y1 = posiciones[id(hijo)]
                ax.plot([x0, x1], [y0, y1], color='#555555', lw=1.6, zorder=1)
                ax.text((x0 + x1) / 2, (y0 + y1) / 2 + 0.08, lado, fontsize=8,
                        color='#888888', ha='center', va='center', zorder=2)

    # Nodos
    for n in nodos:
        x, y = posiciones[id(n)]
        color = COLORES[n.tipo][0]
        ax.text(x, y, n.valor, ha='center', va='center', fontsize=12, fontweight='bold',
                color='white', zorder=3,
                bbox=dict(boxstyle='circle,pad=0.45' if len(n.valor) <= 2 else 'round,pad=0.45',
                          fc=color, ec='black', lw=1.2))

    tipos_presentes = {n.tipo for n in nodos}
    leyenda = [Patch(fc=COLORES[t][0], ec='black', label=COLORES[t][1])
               for t in COLORES if t in tipos_presentes]
    ax.legend(handles=leyenda, loc='upper left', bbox_to_anchor=(1.01, 1), frameon=False)

    ax.set_xlim(-1, len(nodos))
    ax.set_ylim(-(h - 1) - 0.6, 0.6)
    ax.set_title(titulo, fontsize=13, fontweight='bold')
    ax.axis('off')
    plt.tight_layout()
    if guardar_como:
        plt.savefig(guardar_como, dpi=200, bbox_inches='tight')
        print(f"💾 Árbol guardado como '{guardar_como}'")
    plt.show()


def graficar_funcion(raiz, var, rango=(-10, 10), puntos=1000):
    a, b = rango
    if not (isinstance(a, (int, float)) and isinstance(b, (int, float))) or a >= b:
        print("⚠️ Rango inválido: debe ser (mínimo, máximo) con mínimo < máximo.")
        return
    xs = np.linspace(a, b, puntos)
    ys = []
    for x in xs:
        try:
            ys.append(evaluar(raiz, {var: x}))
        except ErrorExpresion:
            ys.append(np.nan)
    ys = np.array(ys, dtype=float)
    if np.all(np.isnan(ys)):
        print(f"⚠️ La función no está definida en ningún punto de [{a}, {b}].")
        return

    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.plot(xs, ys, lw=2, color='#4C72B0', label=f"f({var}) = {infija_con_parentesis(raiz)}")
    ax.axhline(0, color='black', lw=0.8)
    ax.axvline(0, color='black', lw=0.8)
    ax.grid(alpha=0.3)
    ax.set_xlabel(var)
    ax.set_ylabel(f"f({var})")
    ax.set_title("Gráfica de la función (se omiten los puntos fuera del dominio)")
    ax.legend(loc='best', fontsize=9)
    plt.tight_layout()
    plt.show()


print("✅ Graficación lista.")

## Celda 8 · Función principal `analizar()`

Une todo el proceso. Parámetros:

| Parámetro | Para qué sirve | Ejemplo |
|---|---|---|
| `expr` | la expresión a analizar | `"sqrt(2*(x^2)+5*x-3)"` |
| `valores` | valores de las variables para evaluar | `{'x': 2}` |
| `mostrar_pasos` | imprime la tabla de Shunting-Yard | `True` |
| `graficar` | dibuja el árbol | `True` |
| `graficar_f` | grafica la función si tiene una variable | `True` |
| `rango` | intervalo de la gráfica de la función | `(-10, 10)` |
| `guardar_como` | guarda la imagen del árbol | `"arbol.png"` |

Regresa la **raíz** del árbol, o `None` si la expresión no es válida.

In [ ]:
def construir_desde_texto(expr):
    """Proceso completo de texto a árbol. Lanza ErrorExpresion si algo falla."""
    normalizada = normalizar(expr)
    tokens = tokenizar(normalizada)
    tokens, avisos = preprocesar(tokens, normalizada)
    validar_sintaxis(tokens, normalizada)
    postfija, pasos = a_postfija(tokens)
    raiz = construir_arbol(postfija)
    return raiz, normalizada, tokens, postfija, pasos, avisos


def _validar_valores(valores):
    if valores is None:
        return {}
    if not isinstance(valores, dict):
        raise ErrorExpresion("'valores' debe ser un diccionario, por ejemplo {'x': 2}.")
    limpios = {}
    for k, v in valores.items():
        if isinstance(v, bool) or not isinstance(v, (int, float)):
            raise ErrorExpresion(f"El valor de '{k}' debe ser numérico (recibido: {v!r}).")
        if not math.isfinite(v):
            raise ErrorExpresion(f"El valor de '{k}' debe ser un número finito.")
        limpios[str(k).lower()] = float(v)
    return limpios


def analizar(expr, valores=None, mostrar_pasos=True, graficar=True,
             graficar_f=True, rango=(-10, 10), guardar_como=None):
    linea = '═' * 70
    print(linea)
    print(f"📥 Expresión recibida: {expr}")
    print(linea)

    try:
        raiz, normalizada, tokens, postfija, pasos, avisos = construir_desde_texto(expr)
    except ErrorExpresion as err:
        print("❌ Expresión NO válida:")
        print(f"   {err}")
        return None

    print(f"✅ Expresión válida")
    print(f"   Normalizada: {normalizada}")
    for aviso in avisos:
        print(f"   ℹ️  {aviso}")

    print("\n🔤 TOKENS")
    print("   " + '  '.join(f"[{t.valor}]" for t in tokens))

    if mostrar_pasos:
        print("\n🧮 CONVERSIÓN INFIJA → POSTFIJA (Shunting-Yard)\n")
        imprimir_pasos(pasos)

    pre, ino, post = preorden(raiz), inorden(raiz), postorden(raiz)
    print("\n🌳 RECORRIDOS DEL ÁRBOL")
    print(f"   Preorden  (raíz-izq-der, prefija) : {' '.join(pre)}")
    print(f"   Inorden   (izq-raíz-der, infija)  : {' '.join(ino)}")
    print(f"   Postorden (izq-der-raíz, postfija): {' '.join(post)}")
    print(f"   Infija con paréntesis             : {infija_con_parentesis(raiz)}")

    coincide = post == [t.valor for t in postfija]
    print(f"\n🔎 Verificación: postorden == postfija de Shunting-Yard → {'✔ coinciden' if coincide else '✘ NO coinciden'}")

    vars_expr = sorted(variables(raiz))
    print("\n📊 ESTADÍSTICAS")
    print(f"   Nodos: {contar_nodos(raiz)} | Hojas: {contar_hojas(raiz)} | "
          f"Nodos internos: {contar_nodos(raiz) - contar_hojas(raiz)} | Altura: {altura(raiz)} | "
          f"Raíz: '{raiz.valor}'")
    print(f"   Variables: {', '.join(vars_expr) if vars_expr else '(ninguna)'}")

    # Evaluación
    try:
        valores = _validar_valores(valores)
        sobrantes = set(valores) - set(vars_expr)
        if sobrantes:
            print(f"   ⚠️ Se ignoran valores de variables que no aparecen: {', '.join(sorted(sobrantes))}")
        if not vars_expr or valores:
            resultado = evaluar(raiz, valores)
            asignacion = ', '.join(f"{k} = {valores[k]:g}" for k in vars_expr)
            print(f"\n🧾 EVALUACIÓN{' con ' + asignacion if asignacion else ''}: {resultado:.10g}")
    except ErrorExpresion as err:
        print(f"\n⚠️ No se pudo evaluar: {err}")

    if graficar:
        dibujar_arbol(raiz, titulo=f"Árbol de expresión: {normalizada}", guardar_como=guardar_como)
    if graficar_f and len(vars_expr) == 1:
        graficar_funcion(raiz, vars_expr[0], rango=rango)
    return raiz


print("✅ Todo listo. Ejecuta la siguiente celda.")

## Celda 9 · Ejecución con la ecuación de la imagen

$\sqrt{2x^2 + 5x - 3}$, evaluada en $x = 2$:
$\sqrt{2(4) + 10 - 3} = \sqrt{15} \approx 3.873$

👉 **Para usar otra expresión, cambia el texto de `EXPRESION` y vuelve a ejecutar esta celda.**

In [ ]:
EXPRESION = "sqrt(2*(x^2)+5*x-3)"
VALORES = {'x': 2}

arbol = analizar(EXPRESION, valores=VALORES, rango=(-8, 6), guardar_como="arbol_expresion.png")

### Misma ecuación sin la raíz, escrita "a mano" (segunda línea de la imagen)

`2x^2 + 5x - 3` usa multiplicación implícita; el programa la detecta y la convierte en `2*x^2 + 5*x - 3`.

In [ ]:
arbol2 = analizar("2x^2 + 5x - 3", valores={'x': 2}, mostrar_pasos=False, rango=(-5, 3))

## Celda 10 · Prueba de validaciones

Se prueban expresiones correctas e incorrectas para demostrar que cada error se detecta con un mensaje claro.

In [ ]:
CASOS = [
    # (expresión, ¿debe ser válida?)
    ("sqrt(2*(x^2)+5*x-3)", True),
    ("-x^2 + 3*(x - 1)/2", True),
    ("2^3^2", True),
    ("sin(pi/2) + ln(e)", True),
    ("(x+1)(x-1)", True),
    ("√(x)", True),
    ("", False),
    ("2*(x+3", False),
    ("2*x+3)", False),
    ("2*/x", False),
    ("*x+1", False),
    ("x+", False),
    ("()", False),
    ("sqrt x", False),
    ("raiz(x)", False),
    ("2 3 + x", False),
    ("3.4.5 + x", False),
    ("x & 2", False),
    ("y = 2x + 1", False),
]

aciertos = 0
for expr, esperado in CASOS:
    try:
        construir_desde_texto(expr)
        valida, detalle = True, ""
    except ErrorExpresion as err:
        valida, detalle = False, str(err).split('\n')[0]
    ok = valida == esperado
    aciertos += ok
    estado = "VÁLIDA  " if valida else "INVÁLIDA"
    print(f"{'✔' if ok else '✘'} {estado} │ {expr!r:<26} │ {detalle}")

print(f"\nResultado: {aciertos}/{len(CASOS)} casos se comportaron como se esperaba.")

print("\nPruebas de evaluación (errores de dominio):")
for expr, vals in [("sqrt(2*(x^2)+5*x-3)", {'x': 0}),
                   ("1/(x-2)", {'x': 2}),
                   ("ln(x)", {'x': -1}),
                   ("(-8)^(1/3)", {}),
                   ("exp(x)", {'x': 1000})]:
    raiz = construir_desde_texto(expr)[0]
    try:
        print(f"   {expr} con {vals}: {evaluar(raiz, vals):g}")
    except ErrorExpresion as err:
        print(f"   {expr} con {vals}: ⚠️ {err}")

## Celda 11 · Modo interactivo (opcional)

Escribe expresiones una por una. Si la expresión tiene variables, se piden sus valores
(deja vacío para no evaluar). Escribe **`salir`** para terminar.

In [ ]:
def modo_interactivo():
    print("Escribe una expresión (o 'salir' para terminar).")
    print("Ejemplos: sqrt(2*(x^2)+5*x-3) | (a+b)*(a-b) | 3x^2 - 2x + 1 | sin(x)/x\n")
    while True:
        expr = input("➤ Expresión: ").strip()
        if expr.lower() in ('salir', 'exit', 'q'):
            print("👋 Fin del modo interactivo.")
            break
        try:
            raiz = construir_desde_texto(expr)[0]
        except ErrorExpresion as err:
            print(f"❌ {err}\n")
            continue

        valores = {}
        for v in sorted(variables(raiz)):
            while True:
                texto = input(f"   Valor de {v} (Enter para no evaluar): ").strip().replace(',', '.')
                if texto == '':
                    valores = None
                    break
                try:
                    valores[v] = float(texto)
                    break
                except ValueError:
                    print("   ⚠️ Escribe un número válido, por ejemplo 2 o -3.5")
            if valores is None:
                break
        analizar(expr, valores=valores, mostrar_pasos=True)
        print()


modo_interactivo()